# Fernhill Box: exploring the export

A scratchpad for looking at the data. Try things here; the work that is checked goes in the package `fernhill_churn/` and in `answers/`. Each section says which step it belongs to: the cells that import from `fernhill_churn` work once you have written that step's functions.

Run `make data` first. **Kernel → Restart Kernel and Run All Cells** runs everything from the top.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

if Path.cwd().name == "notebooks":
    os.chdir("..")  # work from the project folder, where data/raw and fernhill_churn are
pd.set_option("display.max_columns", 30)
print(Path.cwd())

## Step 1: the subscriptions

One row per subscription. Which columns could you know on the morning of the cutoff, and which were written later?

In [ ]:
subscriptions = pd.read_csv("data/raw/subscriptions.csv", parse_dates=["started_on", "cancelled_on"])
subscriptions.head()

In [ ]:
subscriptions.info()

In [ ]:
from fernhill_churn.frame import CUTOFF, label_table, load_raw

raw = load_raw()
labels = label_table(raw["subscriptions"], CUTOFF)
labels["churned"].value_counts()

## Step 2: profile every table

Missing values, types and distinct values, then a closer look at the columns that look wrong.

In [ ]:
from fernhill_churn.explore import profile

for name, table in raw.items():
    print(name, len(table), "rows")
    display(profile(table))

In [ ]:
raw["subscriptions"]["plan"].value_counts()

In [ ]:
raw["customers"]["age"].describe()

In [ ]:
repeated = raw["customers"][raw["customers"]["customer_id"].duplicated(keep=False)]
repeated.sort_values("customer_id", kind="stable").head(6)  # each repeated customer's two rows, in file order

## Step 4: what a leak looks like

A model given one fact, "this customer has a cancellation reason", and nothing else. Compare its ROC AUC with the first model's.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from fernhill_churn.features import training_table

table = training_table(raw, CUTOFF)
leaky = table.merge(raw["subscriptions"][["customer_id", "cancellation_reason"]], on="customer_id")
leaky["has_reason"] = leaky["cancellation_reason"].notna().astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    leaky[["has_reason"]], leaky["churned"], test_size=0.25, stratify=leaky["churned"], random_state=42
)
model = LogisticRegression().fit(X_train, y_train)
print("ROC AUC with cancellation_reason:", round(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]), 3))

## Steps 7 and 8: after `make train`

The metrics and the profit by threshold your `train()` wrote.

In [ ]:
import json

metrics = Path("reports/metrics.json")
if metrics.is_file():
    print(json.dumps(json.loads(metrics.read_text()), indent=2))
    display(pd.read_csv("reports/profit_by_threshold.csv").iloc[::10])
else:
    print("run make train first")